# DuoT5 Pointwise Proxy Integrated Gradients

This notebook implements a **reference-conditioned pointwise proxy** for `castorini/duot5-base-msmarco`. DuoT5 is natively pairwise, so it does not define a clean standalone `s(q, d)`. To obtain a pointwise-style explanation, we keep a **fixed query-specific reference passage** in `Document1`, place the document of interest in `Document0`, and attribute the scalar target `logit(true)` at the first decoder step.

Methodologically, this is **not a native pointwise DuoT5 score**. It is a pseudo-pointwise proxy: `s*(q, d) = logit_true(q, d, d_ref(q))`. The reference passage is chosen from the same query candidate set, preferring the worst-ranked available passage and falling back to the second-worst if needed.


In [1]:
# -- IMPORTS --
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from captum.attr import IntegratedGradients
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


/Users/evelinalune/Documents/uni/MSc-IS/thesis/ig-thesis/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
model_name = "castorini/duot5-base-msmarco"
out_dir = Path("../outputs_t5")
pair_paths = [
    out_dir / "pairwise_scores.parquet",
    out_dir / "pairwise_scores.pkl",
    out_dir / "pairwise_scores.csv",
]
ranked_results_path = out_dir / "ranked_results.csv"

pointwise_out = out_dir / "attributions_pointwise_proxy.pkl"
pointwise_summary_out = out_dir / "attributions_pointwise_proxy_summary.csv"

n_steps = 100
max_length = 512
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)


In [3]:
def load_pairs(paths):
    for path in paths:
        if not path.exists():
            continue
        if path.suffix == ".parquet":
            try:
                return pd.read_parquet(path)
            except ImportError:
                continue
        if path.suffix in {".pkl", ".pickle"}:
            return pd.read_pickle(path)
        if path.suffix == ".csv":
            return pd.read_csv(path)
    raise FileNotFoundError("No readable DuoT5 pairwise score file was found.")

pairs_df = load_pairs(pair_paths)
if not ranked_results_path.exists():
    raise FileNotFoundError(f"Missing ranked results file: {ranked_results_path}")
ranked_df = pd.read_csv(ranked_results_path)
ranked_df["qid"] = ranked_df["qid"].astype(str)
ranked_df["pid"] = ranked_df["pid"].astype(str)

pairs_df["qid"] = pairs_df["qid"].astype(str)
pairs_df["pid_i"] = pairs_df["pid_i"].astype(str)
pairs_df["pid_j"] = pairs_df["pid_j"].astype(str)

print(f"Loaded {len(pairs_df):,} DuoT5 preference pairs")
print(f"Loaded {len(ranked_df):,} DuoT5 ranked rows")
pairs_df.head()


Loaded 80 DuoT5 preference pairs
Loaded 993 DuoT5 ranked rows


,qid,query,pid_i,passage_i,score_i,pid_j,passage_j,score_j,g_score,correct_pref
0,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,4083953,Cost to Attend. The total cost to attend inclu...,0.384977,0.615020,1
1,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,2281863,"If you're paying for college, you will save li...",0.389508,0.610489,1
2,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,6262988,Undergraduate Tuition. Southern Illinois Unive...,0.232101,0.767897,1
3,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,6650510,"The overall cost for on-campus, in-state stude...",0.828117,0.171880,1
4,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,4872508,Cost of Attendance. Cost of Attendance (COA) i...,0.170587,0.829411,1


In [4]:
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
except Exception as e:
    raise ImportError(
        "Failed to load the DuoT5 tokenizer. Install the required tokenizer dependencies, "
        "for example `pip install sentencepiece protobuf`, restart the kernel, and rerun."
    ) from e

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
model = model.to(device)
model.eval()

embedding_layer = model.get_input_embeddings()
decoder_start_token_id = model.config.decoder_start_token_id
if decoder_start_token_id is None:
    decoder_start_token_id = tokenizer.pad_token_id

true_ids = tokenizer.encode("true", add_special_tokens=False)
false_ids = tokenizer.encode("false", add_special_tokens=False)
if len(true_ids) != 1 or len(false_ids) != 1:
    raise ValueError("Expected 'true' and 'false' to map to single tokens for DuoT5 scoring.")

true_token_id = int(true_ids[0])
false_token_id = int(false_ids[0])

print(f"Loaded: {model_name}")
print(f"Device: {device}")
print(f"decoder_start_token_id: {decoder_start_token_id}")
print(f"true token id: {true_token_id}")
print(f"false token id: {false_token_id}")


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Loaded: castorini/duot5-base-msmarco
Device: mps
decoder_start_token_id: 0
true token id: 1176
false token id: 6136


In [5]:
def build_reference_pool(ranked_rows):
    pool = {}
    for qid, group in ranked_rows.groupby("qid", sort=False):
        # Prefer the worst-ranked passages as reference documents.
        ordered = group.sort_values("rank", ascending=False)
        pool[qid] = [
            {
                "pid": str(row.pid),
                "passage": row.passage,
                "rank": int(row.rank),
                "score": float(row.score),
            }
            for row in ordered.itertuples(index=False)
        ]
    return pool

reference_pool = build_reference_pool(ranked_df)

def choose_reference(qid, excluded_pids=()):
    excluded = {str(pid) for pid in excluded_pids if pid is not None}
    candidates = reference_pool.get(str(qid), [])
    for candidate in candidates:
        if candidate["pid"] not in excluded:
            return candidate
    if not candidates:
        raise ValueError(f"No reference passage available for qid={qid}")
    return candidates[0]

sample_qid = pairs_df.iloc[0]["qid"]
print(f"Sample reference for qid={sample_qid}: rank={choose_reference(sample_qid)['rank']}")


Sample reference for qid=1049774: rank=10


In [6]:
def duo_input(query, doc0, doc1):
    return f"Query: {query} Document0: {doc0} Document1: {doc1} Relevant:"

def ids_to_embeds(input_ids):
    return embedding_layer(input_ids)

def _tok_len(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def tokenize_duo(query, doc0, doc1, max_length=max_length):
    text = duo_input(query, doc0, doc1)
    encoded = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        padding=False,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

    query_prefix = "Query: "
    doc0_prefix = " Document0: "
    doc1_prefix = " Document1: "
    suffix = " Relevant:"

    c1 = query_prefix
    c2 = c1 + query
    c3 = c2 + doc0_prefix
    c4 = c3 + doc0
    c5 = c4 + doc1_prefix
    c6 = c5 + doc1
    c7 = c6 + suffix

    l1 = _tok_len(c1)
    l2 = _tok_len(c2)
    l3 = _tok_len(c3)
    l4 = _tok_len(c4)
    l5 = _tok_len(c5)
    l6 = _tok_len(c6)
    l7 = _tok_len(c7)

    expected_full_len = l7 + 1
    if input_ids.shape[1] != expected_full_len:
        print(
            f"Warning: segment length mismatch for qid input. "
            f"Expected {expected_full_len}, got {input_ids.shape[1]}. "
            f"This can happen due to truncation; segment positions will be clipped."
        )

    query_positions = [pos for pos in range(l1, min(l2, input_ids.shape[1]))]
    doc0_positions = [pos for pos in range(l3, min(l4, input_ids.shape[1]))]
    doc1_positions = [pos for pos in range(l5, min(l6, input_ids.shape[1]))]

    return {
        "text": text,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "tokens": tokens,
        "query_positions": query_positions,
        "doc0_positions": doc0_positions,
        "doc1_positions": doc1_positions,
    }


In [7]:
def forward_duo_true_logit_from_embeds(input_embeds, attention_mask):
    batch_size = input_embeds.shape[0]
    decoder_input_ids = torch.full(
        (batch_size, 1),
        decoder_start_token_id,
        dtype=torch.long,
        device=input_embeds.device,
    )

    outputs = model(
        inputs_embeds=input_embeds,
        attention_mask=attention_mask,
        decoder_input_ids=decoder_input_ids,
    )

    logits = outputs.logits[:, 0, :]
    return logits[:, true_token_id]

def predict_duo_pointwise_proxy(query, doc0, doc1):
    tok = tokenize_duo(query, doc0, doc1)
    decoder_input_ids = torch.full(
        (1, 1),
        decoder_start_token_id,
        dtype=torch.long,
        device=device,
    )

    with torch.no_grad():
        outputs = model(
            input_ids=tok["input_ids"],
            attention_mask=tok["attention_mask"],
            decoder_input_ids=decoder_input_ids,
        )

    logits = outputs.logits[:, 0, :]
    true_logit = logits[:, true_token_id].item()
    false_logit = logits[:, false_token_id].item()
    tf_logits = logits[:, [false_token_id, true_token_id]]
    true_prob = torch.softmax(tf_logits, dim=-1)[:, 1].item()
    margin = (logits[:, true_token_id] - logits[:, false_token_id]).item()
    return true_prob, true_logit, false_logit, margin

def make_baseline_input_ids(input_ids):
    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)
    eos_id = tokenizer.eos_token_id
    if eos_id is not None:
        for pos, token_id in enumerate(input_ids[0].tolist()):
            if token_id == eos_id:
                baseline_ids[0, pos] = eos_id
    return baseline_ids

def make_baseline_embeds(input_ids):
    baseline_ids = make_baseline_input_ids(input_ids)
    return ids_to_embeds(baseline_ids).detach()


In [8]:
def merge_sentencepiece(tokens, scores):
    special_tokens = set(tokenizer.all_special_tokens)
    word_tokens, word_scores = [], []
    current_word, current_score = "", 0.0

    for token, score in zip(tokens, scores):
        if token in special_tokens:
            if current_word:
                word_tokens.append(current_word)
                word_scores.append(current_score)
                current_word, current_score = "", 0.0
            continue

        if token.startswith("▁"):
            if current_word:
                word_tokens.append(current_word)
                word_scores.append(current_score)
            current_word = token.lstrip("▁") or token
            current_score = score
        else:
            current_word += token
            current_score += score

    if current_word:
        word_tokens.append(current_word)
        word_scores.append(current_score)

    return word_tokens, np.array(word_scores)

def aggregate_span(tokens, token_scores, positions):
    span_tokens = [tokens[pos] for pos in positions if pos < len(tokens)]
    span_scores = np.array([token_scores[pos] for pos in positions if pos < len(token_scores)])
    word_tokens, word_scores = merge_sentencepiece(span_tokens, span_scores)
    return {
        "tokens": span_tokens,
        "token_scores": span_scores,
        "word_tokens": word_tokens,
        "word_scores": word_scores,
        "positions": positions,
    }

def aggregate_attributions(attributions, tok):
    token_scores = attributions[0].sum(dim=-1).detach().cpu().numpy()
    full_word_tokens, full_word_scores = merge_sentencepiece(tok["tokens"], token_scores)

    return {
        "tokens": tok["tokens"],
        "token_scores": token_scores,
        "word_tokens": full_word_tokens,
        "word_scores": full_word_scores,
        "query": aggregate_span(tok["tokens"], token_scores, tok["query_positions"]),
        "doc0": aggregate_span(tok["tokens"], token_scores, tok["doc0_positions"]),
        "doc1": aggregate_span(tok["tokens"], token_scores, tok["doc1_positions"]),
    }


In [9]:
def compute_duot5_pointwise_proxy_ig(query, passage, ref_passage):
    tok = tokenize_duo(query, passage, ref_passage)
    input_embeds = ids_to_embeds(tok["input_ids"]).detach()
    baseline_embeds = make_baseline_embeds(tok["input_ids"])

    ig = IntegratedGradients(forward_duo_true_logit_from_embeds)
    attributions, delta = ig.attribute(
        inputs=input_embeds,
        baselines=baseline_embeds,
        additional_forward_args=(tok["attention_mask"],),
        n_steps=n_steps,
        return_convergence_delta=True,
    )

    true_prob, true_logit, false_logit, margin = predict_duo_pointwise_proxy(query, passage, ref_passage)

    return {
        "method": "pointwise_proxy_ig_duot5",
        "input_text": tok["text"],
        "true_prob": float(true_prob),
        "true_logit": float(true_logit),
        "false_logit": float(false_logit),
        "margin": float(margin),
        **aggregate_attributions(attributions, tok),
        "convergence_delta": float(delta.detach().cpu().item()) if torch.is_tensor(delta) else float(delta),
    }


In [10]:
test_row = pairs_df[pairs_df["correct_pref"] == 1].iloc[0]
ref_i = choose_reference(test_row["qid"], excluded_pids=[test_row["pid_i"]])
ref_j = choose_reference(test_row["qid"], excluded_pids=[test_row["pid_j"]])

print(f"Query: {test_row['query']}")
print(f"Passage i: {test_row['passage_i'][:120]}...")
print(f"Reference for i (rank {ref_i['rank']}): {ref_i['passage'][:120]}...")
print(f"Passage j: {test_row['passage_j'][:120]}...")
print(f"Reference for j (rank {ref_j['rank']}): {ref_j['passage'][:120]}...")


Query: cost of attendance eastern illinois university
Passage i: Eastern Illinois University has roughly 8,000 students. Admission is selective. Tuition is approximately $8,550 per year...
Reference for i (rank 10): Cost of Attendance. Cost of Attendance (COA) is a total of all the usual expenses of being a student. The COA sets the m...
Passage j: Cost to Attend. The total cost to attend includes tuition, student fees and expenses for housing, dining, and supplies. ...
Reference for j (rank 10): Cost of Attendance. Cost of Attendance (COA) is a total of all the usual expenses of being a student. The COA sets the m...


In [11]:
test_proxy_i = compute_duot5_pointwise_proxy_ig(test_row["query"], test_row["passage_i"], ref_i["passage"])
test_proxy_j = compute_duot5_pointwise_proxy_ig(test_row["query"], test_row["passage_j"], ref_j["passage"])

test_true_logit_g = test_proxy_i["true_logit"] - test_proxy_j["true_logit"]
test_true_prob_g = test_proxy_i["true_prob"] - test_proxy_j["true_prob"]

print(f"Pseudo-pointwise true logit i: {test_proxy_i['true_logit']:.4f}")
print(f"Pseudo-pointwise true logit j: {test_proxy_j['true_logit']:.4f}")
print(f"Derived pseudo-pointwise g (true-logit diff): {test_true_logit_g:.4f}")
print(f"Derived pseudo-pointwise g (true-prob diff): {test_true_prob_g:.4f}")
print(f"Convergence delta i: {test_proxy_i['convergence_delta']:.6f}")
print(f"Convergence delta j: {test_proxy_j['convergence_delta']:.6f}")

for label, result in [("doc0 / i", test_proxy_i), ("doc0 / j", test_proxy_j)]:
    scores = result["doc0"]["word_scores"]
    if len(scores) == 0:
        print(f"\nNo document words available for {label} after truncation.")
        continue
    top_idx = np.argsort(scores)[::-1][:10]
    print(f"\nTop positive words for {label}:")
    for idx in top_idx:
        print(f"  {result['doc0']['word_tokens'][idx]:<20} {scores[idx]:.4f}")


Pseudo-pointwise true logit i: -11.6956
Pseudo-pointwise true logit j: -16.3335
Derived pseudo-pointwise g (true-logit diff): 4.6378
Derived pseudo-pointwise g (true-prob diff): 0.0753
Convergence delta i: -0.300452
Convergence delta j: -0.330039

Top positive words for doc0 / i:
  bordering            0.0780
  Eastern              0.0767
  has                  0.0634
  while                0.0556
  Illinois             0.0555
  Tuition              0.0459
  residents            0.0254
  Tuition              0.0232
  selective.           0.0201
  Additional           0.0128

Top positive words for doc0 / j:
  fees                 0.1041
  cost                 0.0724
  cost                 0.0697
  the                  0.0663
  cost                 0.0515
  about                0.0452
  our                  0.0386
  1                    0.0297
  expenses             0.0247
  tuition,             0.0093


In [ ]:
attribution_records = []
failed_pairs = []
point_cache = {}

def cache_key(qid, pid, ref_pid):
    return (str(qid), str(pid), str(ref_pid))

for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Computing DuoT5 pointwise proxy IG"):
    try:
        ref_i = choose_reference(row["qid"], excluded_pids=[row["pid_i"]])
        ref_j = choose_reference(row["qid"], excluded_pids=[row["pid_j"]])

        key_i = cache_key(row["qid"], row["pid_i"], ref_i["pid"])
        key_j = cache_key(row["qid"], row["pid_j"], ref_j["pid"])

        if key_i not in point_cache:
            point_cache[key_i] = compute_duot5_pointwise_proxy_ig(row["query"], row["passage_i"], ref_i["passage"])
        if key_j not in point_cache:
            point_cache[key_j] = compute_duot5_pointwise_proxy_ig(row["query"], row["passage_j"], ref_j["passage"])

        pointwise_proxy_i = point_cache[key_i]
        pointwise_proxy_j = point_cache[key_j]

        attribution_records.append({
            "qid": row["qid"],
            "query": row["query"],
            "pid_i": row["pid_i"],
            "pid_j": row["pid_j"],
            "g_score": row["g_score"],
            "correct_pref": row["correct_pref"],
            "ref_pid_i": ref_i["pid"],
            "ref_pid_j": ref_j["pid"],
            "pointwise_proxy_i": pointwise_proxy_i,
            "pointwise_proxy_j": pointwise_proxy_j,
            "pointwise_true_logit_g": float(pointwise_proxy_i["true_logit"] - pointwise_proxy_j["true_logit"]),
            "pointwise_true_prob_g": float(pointwise_proxy_i["true_prob"] - pointwise_proxy_j["true_prob"]),
        })
    except Exception as e:
        failed_pairs.append({
            "qid": row["qid"],
            "pid_i": row["pid_i"],
            "pid_j": row["pid_j"],
            "error": str(e),
        })
        print(f"Error on qid={row['qid']}, pid_i={row['pid_i']}, pid_j={row['pid_j']}: {e}")

print(f"\nSuccessfully attributed: {len(attribution_records)} pairs")
print(f"Unique pointwise proxy computations cached: {len(point_cache)}")
if failed_pairs:
    print(f"Failed: {len(failed_pairs)} pairs")


Python(43361) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Computing DuoT5 pointwise proxy IG:   0%|          | 0/80 [00:00<?, ?it/s]

Error on qid=1049774, pid_i=7185662, pid_j=6262988: MPS backend out of memory (MPS allocated: 19.76 GiB, other allocations: 1.77 MiB, max allowed: 20.13 GiB). Tried to allocate 411.99 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).
Error on qid=1049774, pid_i=7185662, pid_j=4872508: MPS backend out of memory (MPS allocated: 20.12 GiB, other allocations: 1.80 MiB, max allowed: 20.13 GiB). Tried to allocate 387.64 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).
Error on qid=1049791, pid_i=7185644, pid_j=7185650: MPS backend out of memory (MPS allocated: 20.08 GiB, other allocations: 1.81 MiB, max allowed: 20.13 GiB). Tried to allocate 83.50 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).
Error on qid=1049791, pid_i=7185644, pid_

In [ ]:
true_logit_gaps = [record["pointwise_true_logit_g"] for record in attribution_records]
true_prob_gaps = [record["pointwise_true_prob_g"] for record in attribution_records]
deltas_i = [record["pointwise_proxy_i"]["convergence_delta"] for record in attribution_records]
deltas_j = [record["pointwise_proxy_j"]["convergence_delta"] for record in attribution_records]

print("Convergence delta - DuoT5 pointwise proxy IG:")
print(f" i mean={np.mean(deltas_i):.6f}, i max_abs={np.max(np.abs(deltas_i)):.6f}")
print(f" j mean={np.mean(deltas_j):.6f}, j max_abs={np.max(np.abs(deltas_j)):.6f}")

print("\nDerived pseudo-pointwise g from true logits:")
print(f" mean={np.mean(true_logit_gaps):.4f}, min={np.min(true_logit_gaps):.4f}, max={np.max(true_logit_gaps):.4f}")

print("\nDerived pseudo-pointwise g from true probabilities:")
print(f" mean={np.mean(true_prob_gaps):.4f}, min={np.min(true_prob_gaps):.4f}, max={np.max(true_prob_gaps):.4f}")


In [ ]:
def show_top_words(record, side="i", n=10):
    attr = record["pointwise_proxy_i"] if side == "i" else record["pointwise_proxy_j"]
    words = attr["doc0"]["word_tokens"]
    scores = attr["doc0"]["word_scores"]

    top_pos = np.argsort(scores)[::-1][:n]
    top_neg = np.argsort(scores)[:n]

    print(f"Side: doc_{side}")
    print(f"Query: {record['query']}")
    print(f"Original DuoT5 pairwise g_score: {record['g_score']:.4f}")
    print(f"Derived pseudo-pointwise true-logit g: {record['pointwise_true_logit_g']:.4f}")
    print(f"Reference pid: {record['ref_pid_i'] if side == 'i' else record['ref_pid_j']}")
    print(f"Direct true_logit: {attr['true_logit']:.4f}")

    print("\nTop positive document words:")
    for idx in top_pos:
        print(f"  {words[idx]:<20} {scores[idx]:.4f}")

    print("\nTop negative document words:")
    for idx in top_neg:
        print(f"  {words[idx]:<20} {scores[idx]:.4f}")

example = attribution_records[0]
show_top_words(example, side="i", n=10)
print("\n" + "=" * 80 + "\n")
show_top_words(example, side="j", n=10)


In [ ]:
with open(pointwise_out, "wb") as f:
    pickle.dump(attribution_records, f)

print(f"Saved {len(attribution_records)} attribution records -> {pointwise_out}")

summary = pd.DataFrame([
    {
        "qid": record["qid"],
        "pid_i": record["pid_i"],
        "pid_j": record["pid_j"],
        "g_score": record["g_score"],
        "correct_pref": record["correct_pref"],
        "ref_pid_i": record["ref_pid_i"],
        "ref_pid_j": record["ref_pid_j"],
        "pointwise_true_logit_g": record["pointwise_true_logit_g"],
        "pointwise_true_prob_g": record["pointwise_true_prob_g"],
        "pi_true_logit": record["pointwise_proxy_i"]["true_logit"],
        "pj_true_logit": record["pointwise_proxy_j"]["true_logit"],
        "pi_true_prob": record["pointwise_proxy_i"]["true_prob"],
        "pj_true_prob": record["pointwise_proxy_j"]["true_prob"],
        "pi_ig_delta": record["pointwise_proxy_i"]["convergence_delta"],
        "pj_ig_delta": record["pointwise_proxy_j"]["convergence_delta"],
        "doc_i_top_word": record["pointwise_proxy_i"]["doc0"]["word_tokens"][np.argmax(record["pointwise_proxy_i"]["doc0"]["word_scores"])] if len(record["pointwise_proxy_i"]["doc0"]["word_tokens"]) > 0 else "",
        "doc_j_top_word": record["pointwise_proxy_j"]["doc0"]["word_tokens"][np.argmax(record["pointwise_proxy_j"]["doc0"]["word_scores"])] if len(record["pointwise_proxy_j"]["doc0"]["word_tokens"]) > 0 else "",
    }
    for record in attribution_records
])

summary.to_csv(pointwise_summary_out, index=False)
print(f"Saved summary CSV -> {pointwise_summary_out}")
summary.head()
